# Solar AI - rooftop condition classifier MIX retrain (Phase 19-B)

Research only. Attach `edithstark/solar-ai-ground-condition-4class-classifier-v1` (Susan-derived base) and `edithstark/solar-ai-rooftop-condition-topup-v1` (241 AI-reviewed rooftop crops, heavily CLEAN-skewed - domain top-up, NOT a standalone dataset). Accelerator = GPU. Run All.

⚠️ Rooftop labels are AI-visual-review (Claude), **not owner-verified**. ⚠️ Rooftop non-CLEAN examples are extremely few (3 BIRD_DROP, 1 PHYSICAL_DAMAGE, 2 DUSTY total) - this is domain adaptation (this camera/lighting/panel texture), not new-class learning. The Susan-derived test split is the real benchmark.


## 1. Environment


In [ ]:
import sys, platform, subprocess, json, os, time, hashlib, shutil
from pathlib import Path
PINNED = ['torch==2.7.1', 'torchvision==0.22.1']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINNED], check=True)
import torch
assert torch.cuda.is_available(), 'CUDA unavailable.'
import torchvision
ENV = dict(python=platform.python_version(), torch=torch.__version__,
           torchvision=torchvision.__version__, cuda=torch.version.cuda,
           gpu=torch.cuda.get_device_name(0))
print(json.dumps(ENV, indent=2))


## 2. Locate datasets


In [ ]:
EXPECTED_CLASSES = ['BIRD_DROP', 'CLEAN', 'DUSTY', 'PHYSICAL_DAMAGE']
INPUT = Path('/kaggle/input')
base_candidates = sorted({p.parent for p in INPUT.rglob('train') if p.is_dir()
                          and (p/'CLEAN').is_dir() and 'rooftop' not in str(p).lower()})
assert base_candidates, 'Susan-derived classifier dataset not attached'
BASE = base_candidates[0]
roof_candidates = sorted({p.parent for p in INPUT.rglob('train') if p.is_dir()
                          and (p/'CLEAN').is_dir() and 'rooftop' in str(p).lower()})
assert roof_candidates, 'rooftop condition top-up dataset not attached'
ROOF = roof_candidates[0]
print('base (Susan-derived):', BASE)
print('rooftop top-up      :', ROOF)
for split in ('train', 'valid', 'test'):
    classes_here = sorted(p.name for p in (BASE/split).iterdir() if p.is_dir())
    assert classes_here == EXPECTED_CLASSES, f'{split}: {classes_here}'
roof_report = json.loads((ROOF/'build_report.json').read_text())
print(json.dumps(roof_report, indent=1))


## 3. Data loaders - MIX train (base + rooftop oversampled), UNCHANGED base valid/test


In [ ]:
from torch.utils.data import DataLoader, ConcatDataset
from torchvision import datasets, models, transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
INPUT_SIZE = 224
OVERSAMPLE = 10  # matches the Phase 19 panel-detector MIX strategy

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(INPUT_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(INPUT_SIZE), transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(), transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

base_train_ds = datasets.ImageFolder(BASE/'train', transform=train_tf)
CLASSES = base_train_ds.classes
assert CLASSES == EXPECTED_CLASSES
roof_train_ds = datasets.ImageFolder(ROOF/'train', transform=train_tf)
assert roof_train_ds.classes == CLASSES, 'rooftop top-up class order mismatch'
mix_train_ds = ConcatDataset([base_train_ds] + [roof_train_ds] * OVERSAMPLE)

valid_ds = datasets.ImageFolder(BASE/'valid', transform=eval_tf)  # UNCHANGED base valid
test_ds = datasets.ImageFolder(BASE/'test', transform=eval_tf)    # UNCHANGED base test
assert valid_ds.classes == test_ds.classes == CLASSES

roof_all_ds = datasets.ImageFolder(ROOF, transform=eval_tf) if False else None  # per-split below instead
roof_eval = {sp: datasets.ImageFolder(ROOF/sp, transform=eval_tf) for sp in ('train', 'valid', 'test')
             if (ROOF/sp).is_dir()}
for sp, ds in roof_eval.items():
    assert ds.classes == CLASSES

BATCH_SIZE = 32
train_loader = DataLoader(mix_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
roof_loaders = {sp: DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
                for sp, ds in roof_eval.items()}
print(f'base train={len(base_train_ds)}  rooftop train(x{OVERSAMPLE})={len(roof_train_ds)*OVERSAMPLE}  '
      f'mix train={len(mix_train_ds)}  base valid={len(valid_ds)}  base test={len(test_ds)}')
print('rooftop eval sets (small, real-world check only):', {k: len(v) for k, v in roof_eval.items()})


## 4. Train (identical hyperparameters to Phase 16-B, only the train set changed)


In [ ]:
import torch.nn as nn

SEED = 42
EPOCHS = 30
PATIENCE = 6
LR = 5e-4
torch.manual_seed(SEED)
device = torch.device('cuda')

model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(CLASSES))
model.to(device)

counts = [0] * len(CLASSES)
for _, y in base_train_ds.samples:
    counts[y] += 1
for _, y in roof_train_ds.samples:
    counts[y] += OVERSAMPLE
class_weights = torch.tensor([sum(counts) / (len(CLASSES) * max(1, c)) for c in counts],
                             dtype=torch.float32, device=device)
print('mix train class counts:', dict(zip(CLASSES, counts)))

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

def run_epoch(loader, train):
    model.train(train)
    total_loss, total_correct, total_n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if train: optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            if train:
                loss.backward(); optimizer.step()
            total_loss += float(loss.item()) * images.size(0)
            total_correct += int((logits.argmax(1) == labels).sum().item())
            total_n += images.size(0)
    return total_loss / total_n, total_correct / total_n

CKPT = Path('/kaggle/working/condition_mobilenetv2_4class_rooftop_mix_v1.pth')
history, best_val_acc, best_epoch, stale = [], -1.0, -1, 0
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    ep_t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(valid_loader, train=False)  # base valid - unchanged signal
    ep_dt = time.time() - ep_t0
    history.append(dict(epoch=epoch, train_loss=tr_loss, train_acc=tr_acc,
                         val_loss=va_loss, val_acc=va_acc, seconds=ep_dt))
    print(f'epoch {epoch:2d}/{EPOCHS}  train_loss={tr_loss:.4f} train_acc={tr_acc:.3f}  '
          f'val_loss={va_loss:.4f} val_acc={va_acc:.3f}  ({ep_dt:.1f}s)')
    if va_acc > best_val_acc:
        best_val_acc, best_epoch, stale = va_acc, epoch, 0
        torch.save(model.state_dict(), CKPT)
    else:
        stale += 1
        if stale >= PATIENCE:
            print(f'early stopping at epoch {epoch} (best epoch {best_epoch}, val_acc={best_val_acc:.3f})')
            break
train_seconds = time.time() - t0
print(f'training done in {train_seconds:.1f}s - best epoch {best_epoch}, val_acc={best_val_acc:.3f}')


## 5. Evaluate: (a) Susan-derived TEST [real benchmark] (b) rooftop crops [small, real-world check]


In [ ]:
BASELINE = dict(checkpoint='mobilenet_condition_4class_research_v1.pth (Phase 16-B)', test_accuracy=0.752)
model.load_state_dict(torch.load(CKPT, map_location=device))
model.eval()

def evaluate(loader, classes):
    cm = [[0]*len(classes) for _ in classes]
    n_correct = n_total = 0
    with torch.no_grad():
        for images, labels in loader:
            preds = model(images.to(device)).argmax(1).cpu()
            for t, p in zip(labels.tolist(), preds.tolist()):
                cm[t][p] += 1; n_total += 1; n_correct += int(t == p)
    per_class = {}
    for i, cls in enumerate(classes):
        tp = cm[i][i]; fn = sum(cm[i]) - tp; fp = sum(cm[r][i] for r in range(len(classes))) - tp
        per_class[cls] = dict(n=sum(cm[i]), tp=tp, fp=fp, fn=fn,
                              recall=(tp/(tp+fn) if tp+fn else None))
    return dict(accuracy=n_correct/n_total if n_total else None, n=n_total,
                confusion_matrix=cm, class_order=classes, per_class=per_class)

test_result = evaluate(test_loader, CLASSES)
print('SUSAN-DERIVED TEST (real benchmark, apples-to-apples vs Phase 16-B):')
print(f"  accuracy={test_result['accuracy']:.3f}  (baseline {BASELINE['test_accuracy']})  n={test_result['n']}")

rooftop_results = {}
for sp, loader in roof_loaders.items():
    r = evaluate(loader, CLASSES)
    rooftop_results[sp] = r
    print(f"ROOFTOP {sp} (n={r['n']}, SMALL SAMPLE - not a real benchmark): accuracy={r['accuracy']}")
    for cls, m in r['per_class'].items():
        if m['n'] > 0:
            print(f"    {cls}: n={m['n']} recall={m['recall']}")


## 6. Package


In [ ]:
sha = hashlib.sha256(CKPT.read_bytes()).hexdigest()
manifest = dict(
    experiment_name='condition_mobilenetv2_4class_rooftop_mix_v1',
    phase='19-B - rooftop condition classifier MIX retrain',
    model_role='EXPERIMENTAL - domain-adapted condition classifier, NOT production, NOT owner-validated',
    predecessor='condition_mobilenetv2_4class_classifier_v1 (Phase 16-B)',
    classes_index_order=CLASSES,
    baseline_for_comparison=BASELINE,
    oversample=OVERSAMPLE,
    rooftop_label_provenance='AI visual review (Claude), NOT owner-verified',
    rooftop_dataset_kaggle='edithstark/solar-ai-rooftop-condition-topup-v1',
    base_dataset_kaggle='edithstark/solar-ai-ground-condition-4class-classifier-v1',
    hyperparameters=dict(epochs_requested=EPOCHS, epochs_run=len(history), batch_size=BATCH_SIZE,
                         lr=LR, optimizer='AdamW', patience=PATIENCE, seed=SEED, input_size=INPUT_SIZE),
    best_epoch=best_epoch, best_val_accuracy=best_val_acc, training_seconds=train_seconds,
    history=history,
    susan_test_set=test_result,
    rooftop_realworld_check=rooftop_results,
    checkpoint_sha256=sha, environment=ENV,
    not_production_validated=True,
)
OUT = Path('/kaggle/working/output'); OUT.mkdir(exist_ok=True)
shutil.copy2(CKPT, OUT / CKPT.name)
(OUT / 'training_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
print('checkpoint sha256:', sha)
print('Artifacts:', sorted(p.name for p in OUT.iterdir()))


## 7. Retrieve
Download `/kaggle/working/output/`. Save the checkpoint as `weights/mobilenet_condition_4class_rooftop_mix_v1.pth` (gitignored). Compare `susan_test_set.accuracy` against 0.752 for the real regression check; treat `rooftop_realworld_check` as illustrative only given its tiny per-class counts.
